# WELL Generator Equidistribution

WELL (Well Equidistributed Long-period Linear) generators, implemented
as carry-type generators. We test WELL19937 and WELL44497 with and
without tempering, using the lattice method.

In [ ]:
from regpoly.core.generator import Generator
from regpoly.core.transformation import Transformation
from regpoly.core.combination import Combination
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_DUALLATTICE
)

INT_MAX = 2**31 - 1

## 1. WELL19937a (without tempering)

Carry generator with $w=32$, $r=624$, $p=31$, $k = 32 \times 624 - 31 = 19937$.

In [ ]:
well19937_params = {
    "w": 32, "r": 624, "p": 31,
    "m1": 70, "m2": 179, "m3": 449,
    # Paper Table II (WELL19937a): T0..T7 = M3(-25), M3(27), M2(9), M3(1),
    # M1, M3(-9), M3(-21), M3(21).
    "matrices": {
        "T0": {"M": 3, "t": -25},
        "T1": {"M": 3, "t":  27},
        "T2": {"M": 2, "t":   9},
        "T3": {"M": 3, "t":   1},
        "T4": {"M": 1},
        "T5": {"M": 3, "t":  -9},
        "T6": {"M": 3, "t": -21},
        "T7": {"M": 3, "t":  21},
    },
}

gen_19937 = Generator.create("WELLRNG", 32, **well19937_params)
print(f"{gen_19937.name()}: k={gen_19937.k}, L={gen_19937.L}")
print(gen_19937.display())

In [ ]:
comb = Combination(J=1, Lmax=32)
comb.components[0].add_gen(gen_19937)
comb.reset()

test = EquidistributionTest(L=32, delta=[INT_MAX]*33,
                            mse=INT_MAX, method=METHOD_DUALLATTICE)
result = test.run(comb)

print(f"WELL19937a (no tempering): dimension gaps sum = {result.se}")
print(result.display_table(comb, 'l')[0])

## 2. WELL19937c (with tempering)

Same generator with Matsumoto-Kurita type I tempering.

In [ ]:
temper = Transformation.create("tempMK",
    w=32, eta=7, mu=15,
    b=0xe46e1700, c=0x9b868000)

comb_t = Combination(J=1, Lmax=32)
comb_t.components[0].add_gen(gen_19937)
comb_t.components[0].add_trans(temper)
comb_t.reset()

result_t = test.run(comb_t)

print(f"WELL19937c (with tempering): dimension gaps sum = {result_t.se}")
print(result_t.display_table(comb_t, 'l')[0])

## 3. WELL44497a (without tempering)

Carry generator with $w=32$, $r=1391$, $p=15$, $k = 32 \times 1391 - 15 = 44497$.

In [ ]:
well44497_params = {
    "w": 32, "r": 1391, "p": 15,
    "m1": 23, "m2": 481, "m3": 229,
    # Paper Table II (WELL44497a): T0..T7 = M3(-24), M3(30), M3(-10),
    # M2(-26), M1, M3(20), M6(9, 14, 5, a7), M1; a7 = 0xb729fcec.
    "matrices": {
        "T0": {"M": 3, "t": -24},
        "T1": {"M": 3, "t":  30},
        "T2": {"M": 3, "t": -10},
        "T3": {"M": 2, "t": -26},
        "T4": {"M": 1},
        "T5": {"M": 3, "t":  20},
        "T6": {"M": 6, "q":  9, "t": 14, "s": 5, "a": 0xb729fcec},
        "T7": {"M": 1},
    },
}

gen_44497 = Generator.create("WELLRNG", 32, **well44497_params)
print(f"{gen_44497.name()}: k={gen_44497.k}, L={gen_44497.L}")
print(gen_44497.display())

In [ ]:
comb44 = Combination(J=1, Lmax=32)
comb44.components[0].add_gen(gen_44497)
comb44.reset()

result44 = test.run(comb44)

print(f"WELL44497a (no tempering): dimension gaps sum = {result44.se}")
print(result44.display_table(comb44, 'l')[0])

## 4. Comparison

In [ ]:
print(f"{'Generator':<25}  {'k':>6}  {'Gaps Sum':>10}")
print("-" * 47)
print(f"{'WELL19937a (raw)':<25}  {19937:>6}  {result.se:>10}")
print(f"{'WELL19937c (tempered)':<25}  {19937:>6}  {result_t.se:>10}")
print(f"{'WELL44497a (raw)':<25}  {44497:>6}  {result44.se:>10}")